## 1. Veri Hazırlama

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, recall_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay,
                             roc_curve, auc, classification_report)

# Veriyi oku
column_names = ['Class', 'Alcohol', 'Malic acid', 'Ash', 'Alcalinity of ash',
                'Magnesium', 'Total phenols', 'Flavanoids', 'Nonflavanoid phenols',
                'Proanthocyanins', 'Color intensity', 'Hue', 'OD280/OD315', 'Proline']

df = pd.read_csv('wine/wine.data', names=column_names)
print(f"Veri seti boyutu: {df.shape}")
print(f"Sınıf dağılımı:\n{df['Class'].value_counts().sort_index()}")
df.head()

In [ ]:
# Özellik ve hedef değişken ayrımı
X = df.drop('Class', axis=1)
y = df['Class']

# %80 eğitim - %20 test bölme
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print(f"Eğitim seti: {X_train.shape[0]} örnek")
print(f"Test seti:   {X_test.shape[0]} örnek")

## 2. Ham Veri ve Standardize Veri Karşılaştırması

In [ ]:
# --- Ham Veri ile Model ---
model_ham = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42)
model_ham.fit(X_train, y_train)
acc_ham = accuracy_score(y_test, model_ham.predict(X_test))
print(f"Ham Veri Accuracy: %{acc_ham*100:.2f}")

# --- Standardize Veri ile Model ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model_std = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42)
model_std.fit(X_train_scaled, y_train)
acc_std = accuracy_score(y_test, model_std.predict(X_test_scaled))
print(f"Standardize Veri Accuracy: %{acc_std*100:.2f}")

In [ ]:
# Loss eğrilerini karşılaştırmalı çiz
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(model_ham.loss_curve_, color='red', label=f'Ham Veri (Acc: %{acc_ham*100:.1f})')
axes[0].set_title('Ham Veri - Loss Eğrisi')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(True)

axes[1].plot(model_std.loss_curve_, color='green', label=f'Standardize (Acc: %{acc_std*100:.1f})')
axes[1].set_title('Standardize Veri - Loss Eğrisi')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(True)

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("📝 YORUM: Standardizasyonun Etkisi")
print("="*70)
print(f"""Ham veride özellikler farklı ölçeklerde olduğundan (örn. Proline 0-1700,
Hue 0-2 aralığında), gradient descent büyük ölçekli özelliklere doğru
sapma gösterir ve öğrenme kararsız/yavaş olur.

Standardizasyon sonrası tüm özellikler ortalama=0, std=1 olacak şekilde
dönüştürüldüğünde, loss eğrisi çok daha hızlı ve kararlı bir şekilde
düşmektedir. Accuracy farkı: %{acc_ham*100:.1f} → %{acc_std*100:.1f}

Sonuç: Yapay sinir ağlarında veri standardizasyonu neredeyse zorunludur.
Gradient tabanlı optimizasyon algoritmalarının verimli çalışması için
özelliklerin aynı ölçekte olması gerekir.""")

## 3. Aktivasyon Fonksiyonları ve Learning Rate Analizi

In [ ]:
# --- 3a. Sigmoid vs ReLU Karşılaştırması ---
activations = {'Sigmoid (logistic)': 'logistic', 'ReLU': 'relu'}

fig, ax = plt.subplots(figsize=(10, 5))
colors = {'Sigmoid (logistic)': 'blue', 'ReLU': 'orange'}

for name, act in activations.items():
    model = MLPClassifier(hidden_layer_sizes=(64, 32), activation=act,
                          max_iter=500, random_state=42)
    model.fit(X_train_scaled, y_train)
    acc = accuracy_score(y_test, model.predict(X_test_scaled))
    ax.plot(model.loss_curve_, color=colors[name], label=f'{name} (Acc: %{acc*100:.1f})')
    print(f"{name:25s} → Accuracy: %{acc*100:.2f}")

ax.set_title('Aktivasyon Fonksiyonu Karşılaştırması (Standardize Veri)')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.legend(); ax.grid(True)
plt.tight_layout(); plt.show()

print("\n📝 YORUM: Vanishing Gradient Problemi")
print("-"*50)
print("""Sigmoid fonksiyonunun türevi maksimum 0.25'tir. Derin ağlarda geri yayılım
sırasında bu küçük türevler çarpılarak 'vanishing gradient' (kaybolan gradyan)
problemine yol açar — ilk katmanlar neredeyse hiç öğrenemez.

ReLU ise pozitif bölgede türevi her zaman 1 olduğundan gradyanlar korunur ve
daha hızlı yakınsama sağlar. Ancak negatif bölgede türev 0 olduğundan 'dying
ReLU' riski vardır. Genel olarak ReLU, modern ağlarda tercih edilir.""")

In [ ]:
# --- 3b. Learning Rate Analizi ---
learning_rates = [1.0, 0.001, 1e-7]
lr_colors = ['red', 'green', 'purple']

fig, ax = plt.subplots(figsize=(10, 6))

for lr, color in zip(learning_rates, lr_colors):
    model_lr = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500,
                             learning_rate_init=lr, random_state=42)
    model_lr.fit(X_train_scaled, y_train)
    acc = accuracy_score(y_test, model_lr.predict(X_test_scaled))
    ax.plot(model_lr.loss_curve_, color=color,
            label=f'LR={lr} (Acc: %{acc*100:.1f})')
    print(f"LR={lr:<10} → Accuracy: %{acc*100:.2f}  |  Son Loss: {model_lr.loss_curve_[-1]:.4f}")

ax.set_title('Learning Rate Karşılaştırması')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.legend(); ax.grid(True)
plt.tight_layout(); plt.show()

print("\n📝 YORUM: Learning Rate Etkisi")
print("-"*50)
print("""• LR = 1.0 (Çok yüksek): Loss eğrisi kararsızdır, minimum noktayı sürekli
  aşar (overshooting). Model yakınsamakta zorlanır veya hiç yakınsamaz.

• LR = 0.001 (Optimum): Loss eğrisi düzgün ve hızlı bir şekilde düşer.
  Model kararlı bir şekilde öğrenir ve iyi bir doğruluğa ulaşır.

• LR = 1e-7 (Çok düşük): Loss neredeyse hiç düşmez. Öğrenme aşırı yavaştır
  ve 500 epoch yeterli değildir. Pratikte bu durum 'öğrenme yok' demektir.

Sonuç: Learning rate, modelin en kritik hiperparametrelerinden biridir.
Ne çok büyük ne çok küçük olmalıdır.""")

## 4. Kapsamlı Performans Değerlendirme
En iyi model: Standardize veri + varsayılan ayarlar

In [ ]:
# En iyi modeli eğit (standardize, relu, lr=0.001)
best_model = MLPClassifier(hidden_layer_sizes=(64, 32), activation='relu',
                           learning_rate_init=0.001, max_iter=500, random_state=42)
best_model.fit(X_train_scaled, y_train)
y_pred = best_model.predict(X_test_scaled)

# Metrikler
acc = accuracy_score(y_test, y_pred)
recall = recall_score(y_test, y_pred, average='macro')
f1 = f1_score(y_test, y_pred, average='macro')

print("="*50)
print("    PERFORMANS METRİKLERİ (En İyi Model)")
print("="*50)
print(f"  Accuracy  (Doğruluk)  : %{acc*100:.2f}")
print(f"  Recall    (Duyarlılık): %{recall*100:.2f}")
print(f"  F1-Score              : %{f1*100:.2f}")
print("="*50)
print("\nDetaylı Sınıflandırma Raporu:")
print(classification_report(y_test, y_pred, target_names=['Sınıf 1', 'Sınıf 2', 'Sınıf 3']))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(7, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Sınıf 1', 'Sınıf 2', 'Sınıf 3'])
disp.plot(ax=ax, cmap='Blues', values_format='d')
ax.set_title('Confusion Matrix (Karmaşıklık Matrisi)')
plt.tight_layout(); plt.show()

print("\n📝 YORUM: Confusion Matrix Analizi")
print("-"*50)
print(f"""Köşegen üzerindeki değerler doğru sınıflandırmaları gösterir.
Köşegen dışındaki değerler hangi sınıfların birbiriyle karıştırıldığını
ortaya koyar. Yanlış sınıflandırmalar genellikle kimyasal özellikleri
birbirine yakın olan şarap sınıfları arasında gerçekleşir.""")

In [ ]:
# ROC Curve & AUC
classes = sorted(y.unique())
y_test_bin = label_binarize(y_test, classes=classes)
y_score = best_model.predict_proba(X_test_scaled)

fig, ax = plt.subplots(figsize=(8, 6))
colors_roc = ['blue', 'green', 'red']

for i, (cls, color) in enumerate(zip(classes, colors_roc)):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_score[:, i])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=2,
            label=f'Sınıf {cls} (AUC = {roc_auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Rastgele (AUC = 0.5)')
ax.set_xlim([0, 1]); ax.set_ylim([0, 1.05])
ax.set_xlabel('False Positive Rate (Yanlış Pozitif Oranı)')
ax.set_ylabel('True Positive Rate (Doğru Pozitif Oranı)')
ax.set_title('ROC Eğrisi (Her Sınıf İçin)')
ax.legend(loc='lower right'); ax.grid(True)
plt.tight_layout(); plt.show()

print("\n📝 YORUM: ROC & AUC Analizi")
print("-"*50)
print("""AUC değeri 1.0'a ne kadar yakınsa model o kadar başarılıdır.
0.5 değeri rastgele tahmine eşittir. Tüm sınıflar için AUC değerlerinin
yüksek olması, modelin her sınıfı başarılı biçimde ayırt edebildiğini gösterir.""")

## 5. Genel Değerlendirme

### 🔹 Veri Standardizasyonunun Önemi
Yapay sinir ağları, gradient descent tabanlı optimizasyon kullanır. Özellikler farklı ölçeklerde olduğunda (örneğin Proline 0–1700, Hue 0–2), büyük ölçekli özellikler loss fonksiyonunu domine eder ve gradyanlar dengesiz olur. StandardScaler ile her özellik ortalama=0, standart sapma=1 olacak şekilde dönüştürüldüğünde, optimizasyon çok daha hızlı ve kararlı bir şekilde yakınsar. Bu deneyde ham veri ile düşük doğruluk elde edilirken, standardize veri ile doğruluk önemli ölçüde artmıştır.

### 🔹 Learning Rate'in Etkisi
Learning rate, her adımda ağırlıkların ne kadar güncelleneceğini belirler:
- **Çok yüksek LR (1.0):** Optimum noktayı sürekli aşar (overshooting), loss salınım yapar ve model yakınsamaz.
- **Uygun LR (0.001):** Loss düzgün ve hızlı bir şekilde düşer, model optimal noktaya ulaşır.
- **Çok düşük LR (1e-7):** Adımlar çok küçüktür, loss neredeyse hiç azalmaz ve model pratik sürede öğrenemez.

Bu nedenle learning rate, modelin en kritik hiperparametresidir ve dikkatle ayarlanmalıdır.

### 🔹 Aktivasyon Fonksiyonlarının Rolü
Aktivasyon fonksiyonları, sinir ağlarına doğrusal olmayan öğrenme yeteneği kazandırır:
- **Sigmoid (Logistic):** Çıktıyı [0,1] aralığına sıkıştırır. Ancak türevi maksimum 0.25 olduğundan, derin ağlarda katmanlar arası gradyanlar çarpılarak sıfıra yaklaşır (*vanishing gradient*). Bu durum öğrenmeyi yavaşlatır.
- **ReLU:** Pozitif girdiler için f(x)=x, negatif girdiler için f(x)=0 döner. Türevi pozitif bölgede 1 olduğundan gradyanlar korunur ve daha hızlı öğrenme sağlanır. Modern ağlarda en yaygın tercih edilen aktivasyon fonksiyonudur.

### 🔹 Yapay Sinir Ağlarının (ANN) Temel Çalışma Mantığı
Yapay sinir ağı, biyolojik sinir hücrelerinden esinlenen bir makine öğrenmesi modelidir. Temel çalışma adımları:
1. **İleri Yayılım (Forward Propagation):** Giriş verileri ağırlıklarla çarpılır, bias eklenir ve aktivasyon fonksiyonundan geçirilerek çıktı üretilir.
2. **Loss Hesaplama:** Üretilen çıktı ile gerçek değer arasındaki fark (hata) bir kayıp fonksiyonu ile ölçülür.
3. **Geri Yayılım (Backpropagation):** Zincir kuralı kullanılarak her ağırlığın hataya katkısı (gradyan) hesaplanır.
4. **Ağırlık Güncelleme:** Gradyan iniş (gradient descent) ile ağırlıklar, hatayı azaltacak yönde güncellenir.

Bu döngü her epoch'ta tekrarlanarak model giderek daha iyi tahminler yapmayı öğrenir. Katman sayısı ve nöron sayısı arttıkça ağ daha karmaşık ilişkileri modelleyebilir, ancak aşırı öğrenme (overfitting) riski de artar.